In [1]:
# =====================================================
# 🔧 0. ENV (CUDA hatalarını kapat)
# =====================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [2]:
import pandas as pd
import numpy as np
import tensorflow_decision_forests as tfdf

E0000 00:00:1776887522.023474      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776887522.096636      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776887522.689617      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776887522.689663      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776887522.689667      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776887522.689669      16 computation_placer.cc:177] computation placer already registered. Please check linka

In [3]:
# 1. VERİLERİ YÜKLE
train_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

In [4]:
# 2. AYKIRI DEĞER TEMİZLİĞİ (Skoru doğrudan iyileştirir)
train_df = train_df.drop(train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)].index)

In [5]:
# 3. FEATURE ENGINEERING (Özellikleri zenginleştirme)
def create_features(df):
    # Toplam alan
    df["TotalSF"] = df["GrLivArea"] + df["TotalBsmtSF"]
    # Toplam banyo
    df["TotalBath"] = df["FullBath"] + (0.5 * df["HalfBath"]) + df["BsmtFullBath"] + (0.5 * df["BsmtHalfBath"])
    # Evin yaşı
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    # Sütun isimlerindeki sorunlu karakterleri temizle
    df.columns = [c.replace(".", "_").replace(" ", "_") for c in df.columns]
    return df

train_df = create_features(train_df)
test_df = create_features(test_df)

In [6]:
# Hedef değişkeni logaritmik yap (Yarışma kuralı)
train_df["SalePrice"] = np.log1p(train_df["SalePrice"])

# Id'leri sonra kullanmak için sakla
test_ids = test_df["Id"]

# 4. DATASETLERİ OLUŞTUR (Id'yi her iki tarafta da DROP ediyoruz)
# Bu kısım hatayı çözen kritik nokta: İki tarafta da 'Id' olmayacak!
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_df.drop("Id", axis=1), 
    label="SalePrice", 
    task=tfdf.keras.Task.REGRESSION
)

test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_df.drop("Id", axis=1), 
    task=tfdf.keras.Task.REGRESSION
)

In [7]:
# 5. MODELİ GÜNCELLE (Daha derin öğrenme için)
model = tfdf.keras.GradientBoostedTreesModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=2000,          # Ağaç sayısını artırdık
    shrinkage=0.02,          # Öğrenme hızını düşürdük (daha hassas tahmin)
    max_depth=6,             # Karmaşıklığı kontrol altında tutuyoruz
    # hyperparameter_template yerine manuel ayarlar daha çok kontrol sağlar
    growing_strategy="BEST_FIRST_GLOBAL",
    min_examples=5,
    l1_regularization=0.05,  # Gereksiz özellikleri baskılar
    l2_regularization=0.05   # Aşırı öğrenmeyi engeller
)

# 6. EĞİTİM (Aynı şekilde devam)
model.fit(train_ds)

Use /tmp/tmpvux8jref as temporary training directory
Reading training dataset...


W0000 00:00:1776887551.013361      16 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1776887551.014049      16 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1776887551.014061      16 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


Training dataset read in 0:00:04.509105. Found 1458 examples.
Training model...


I0000 00:00:1776887555.561100      16 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1776887555.561141      16 kernel.cc:783] Collect training examples
I0000 00:00:1776887555.561154      16 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: NUMERICAL
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false

I0000 00:00:1776887555.562286      16 kernel.cc:401] Number of batches: 2
I0000 00:00:1776887555.562311      16 kernel.cc:402] Number of examples: 1458
I0000 00:00:1776887555.565536      16 data_spec_inference.cc:354] 1 item(s) have been pruned (i.e. they are considered out of dictionary) for the column BsmtCond (3 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
I0000 00:00:1776887555.565605      16 data_spec_inference.cc:354] 1 item(s) have been prun

Model trained in 0:00:13.544121
Compiling model...


I0000 00:00:1776887569.083570      16 quick_scorer_extended.cc:927] The binary was compiled without AVX2 support, but your CPU supports it. Enable it for faster model inference.
I0000 00:00:1776887569.089305      16 abstract_model.cc:1439] Engine "GradientBoostedTreesQuickScorerExtended" built


Model compiled.


In [8]:
# 6. EĞİTİM
model.fit(train_ds)

# 7. TAHMİN VE GERİ DÖNÜŞÜM
preds = model.predict(test_ds)
# Logaritmik fiyattan gerçek fiyata dön (exp)
final_preds = np.expm1(preds.flatten())
# Negatif değer ihtimaline karşı clip
final_preds = np.clip(final_preds, 0, None)

Reading training dataset...
Training dataset read in 0:00:00.085629. Found 1458 examples.
Training model...


I0000 00:00:1776887570.978865      16 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1776887570.978906      16 kernel.cc:783] Collect training examples
I0000 00:00:1776887570.978921      16 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: NUMERICAL
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false

I0000 00:00:1776887570.979416      16 kernel.cc:401] Number of batches: 4
I0000 00:00:1776887570.979438      16 kernel.cc:402] Number of examples: 1458
I0000 00:00:1776887570.982570      16 data_spec_inference.cc:354] 1 item(s) have been pruned (i.e. they are considered out of dictionary) for the column BsmtCond (3 item(s) left) because min_value_count=5 and max_number_of_unique_values=2000
I0000 00:00:1776887570.982639      16 data_spec_inference.cc:354] 1 item(s) have been prun

Model trained in 0:00:00.390807
Compiling model...


I0000 00:00:1776887571.181393    1627 kernel.cc:926] Export model in log directory: /tmp/tmpvux8jref with prefix 9690036a2d1c472d
I0000 00:00:1776887571.199939    1627 kernel.cc:944] Save model in resources
I0000 00:00:1776887571.201107      16 abstract_model.cc:921] Model self evaluation:
Number of predictions (with weights): 1
Task: REGRESSION
Loss (SQUARED_ERROR): 0.1318

RMSE: 0.363043
Default RMSE: : 0



Model compiled.
2/2 [==============================] - 1s 15ms/step


In [9]:
# 8. SUBMISSION OLUŞTUR
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_preds
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("İşlem tamam! submission.csv dosyasını indirebilirsin.")

İşlem tamam! submission.csv dosyasını indirebilirsin.
